# Effect of resizing and JPEG conversion

The model preprocessor resizes every image to 512 x 512, so sending larger images only costs upload bandwidth. This notebook measures what happens when the images are downscaled, and converted to JPEG, before being sent to the API: how much smaller the payload gets, how the predictions change (IoU against the reference masks), and whether the API time improves.

Both are options of `hf_api.request_segmentation`, disabled by default: `resize=True` downscales the image to fit in 512 x 512 and keeps its format (a PNG stays a lossless PNG), `to_jpeg=True` re-encodes it as JPEG (quality 90). `scripts/segment_images.py` exposes them as `--resize` and `--jpeg`.

The notebook runs `scripts/segment_images.py` on the 50 images with four configurations, which makes 200 API calls and takes a few minutes:

| Run | Script options | Purpose |
|---|---|---|
| `original` | none, original bytes | baseline |
| `png_resize` | `--resize` | effect of the size alone, lossless |
| `jpeg_resize` | `--resize --jpeg` | effect of the size and the JPEG compression |
| `png_resize_rerun` | `--resize` | same as `png_resize`, to check that the model is deterministic |

Masks and log of each run are saved to `data/experiments/<run>/`.

## Configuration

* `RUN_SCRIPT`: `True` runs the script and replaces the results of the previous execution. It needs the `HF_TOKEN` environment variable, or a `.env` file in the root of the project. `False` analyses the results already in `data/experiments/`.

In [ ]:
import csv
import shutil
import statistics
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
from PIL import Image

from oc_p02_fashion_segmentation.constants import (
    CLOTHING_CLASS_IDS,
    ID_TO_CLASS,
    NUM_CLASSES,
)
from oc_p02_fashion_segmentation.hf_api import prepare_image_for_api
from oc_p02_fashion_segmentation.metrics import (
    intersection_and_union,
    iou_from_counts,
)

RUN_SCRIPT = False

PROJECT_DIR = Path("..").resolve()
SCRIPT = PROJECT_DIR / "scripts" / "segment_images.py"
IMAGE_DIR = PROJECT_DIR / "data" / "input" / "IMG"
REFERENCE_DIR = PROJECT_DIR / "data" / "input" / "Mask"
EXPERIMENTS_DIR = PROJECT_DIR / "data" / "experiments"

RUNS = {
    "original": [],
    "png_resize": ["--resize"],
    "jpeg_resize": ["--resize", "--jpeg"],
    "png_resize_rerun": ["--resize"],
}
PAYLOAD_OPTIONS = {
    "original": {},
    "png_resize": {"resize": True},
    "jpeg_resize": {"resize": True, "to_jpeg": True},
}

image_paths = sorted(IMAGE_DIR.glob("*.png"))
print(f"{len(image_paths)} images")

## Run the segmentation script

Each run starts from an empty folder, and `--all` is not needed since no mask exists yet. The script stops with an error if a request keeps failing, after its own retries.

In [ ]:
def run_script(name, options):
    run_dir = EXPERIMENTS_DIR / name
    shutil.rmtree(run_dir, ignore_errors=True)
    command = [
        sys.executable,
        str(SCRIPT),
        "--output-dir",
        str(run_dir / "masks"),
        "--log-file",
        str(run_dir / "log.csv"),
        *options,
    ]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    print(f"{name:<18}" + " ".join(completed.stdout.strip().splitlines()[-1:]))
    if completed.returncode != 0:
        print(completed.stderr[-1000:])
        raise RuntimeError(f"The script failed for the run {name}")


if RUN_SCRIPT:
    for name, options in RUNS.items():
        run_script(name, options)

## Payload size

Size of what is sent to the API, with each combination of options. The images are 400 x 600, so only the height exceeds 512.

In [ ]:
payload_kb = {
    name: statistics.mean(
        len(prepare_image_for_api(path, **options)[0]) for path in image_paths
    )
    / 1000
    for name, options in PAYLOAD_OPTIONS.items()
}
for name, size in payload_kb.items():
    print(
        f"{name:<12}{size:>6.0f} KB  ({size / payload_kb['original']:.0%} of the original)"
    )

## IoU against the reference masks

The pixel counts are summed over the 50 images before computing one IoU per class, then the mean over the classes gives the dataset mIoU. The per-image mIoU is the mean over the classes present in the image, averaged over the images.

The main metric is the **clothes mIoU**, the mean of the IoU of the clothing classes (upper-clothes, skirt, pants, dress), computed the same two ways: over the dataset, and by image then averaged.

In [ ]:
def load_mask(path):
    return np.array(Image.open(path))


def mask_dir(name):
    return EXPERIMENTS_DIR / name / "masks"


def evaluate(name):
    """Return the dataset IoU by class and the IoU by class of each image."""
    intersection = np.zeros(NUM_CLASSES, dtype=np.int64)
    union = np.zeros(NUM_CLASSES, dtype=np.int64)
    per_image = []
    for path in sorted(mask_dir(name).glob("*.png")):
        i, u = intersection_and_union(
            load_mask(REFERENCE_DIR / path.name), load_mask(path)
        )
        intersection += i
        union += u
        per_image.append(iou_from_counts(i, u))
    return iou_from_counts(intersection, union), np.array(per_image)


def mean_iou(iou, class_ids=slice(None)):
    """Mean IoU of the classes, ignoring the classes absent from both masks."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # No class at all.
        return np.nanmean(iou[..., class_ids], axis=-1)


results = {name: evaluate(name) for name in PAYLOAD_OPTIONS}

print(f"{'class':<15}" + "".join(f"{name:>13}" for name in results))
for class_id in range(NUM_CLASSES):
    print(
        f"{ID_TO_CLASS[class_id]:<15}"
        + "".join(f"{iou[class_id]:>13.4f}" for iou, _ in results.values())
    )
print(
    f"{'dataset mIoU':<15}"
    + "".join(f"{np.nanmean(iou):>13.4f}" for iou, _ in results.values())
)
print(
    f"{'dataset clothes':<15}"
    + "".join(
        f"{mean_iou(iou, CLOTHING_CLASS_IDS):>13.4f}" for iou, _ in results.values()
    )
)
print(
    f"{'per-image mIoU':<15}"
    + "".join(f"{mean_iou(by_image).mean():>13.4f}" for _, by_image in results.values())
)
print(
    f"{'per-image clothes':<15}"
    + "".join(
        f"{np.nanmean(mean_iou(by_image, CLOTHING_CLASS_IDS)):>13.4f}"
        for _, by_image in results.values()
    )
)

### Image by image and pixel by pixel

For each resized run, compared with the original run: on how many images the clothes mIoU is better or worse, and the share of pixels whose class changed.

In [ ]:
mask_names = [path.name for path in sorted(mask_dir("original").glob("*.png"))]
original_per_image = mean_iou(results["original"][1], CLOTHING_CLASS_IDS)

for name in ("png_resize", "jpeg_resize"):
    difference = mean_iou(results[name][1], CLOTHING_CLASS_IDS) - original_per_image
    changed = [
        (load_mask(mask_dir(name) / n) != load_mask(mask_dir("original") / n)).mean()
        for n in mask_names
    ]
    print(
        f"{name:<12} better on {(difference > 1e-9).sum()}, worse on {(difference < -1e-9).sum()}, "
        f"same on {(abs(difference) <= 1e-9).sum()} images; "
        f"pixels changed: mean {np.mean(changed):.2%}, max {max(changed):.2%}"
    )

## Is the model deterministic?

Without this, the differences above could be noise between two runs. `png_resize_rerun` has the same options and code as `png_resize`, and the masks are compared pixel by pixel.

In [ ]:
identical = sum(
    np.array_equal(
        load_mask(mask_dir("png_resize_rerun") / n),
        load_mask(mask_dir("png_resize") / n),
    )
    for n in mask_names
)
print(f"{identical}/{len(mask_names)} masks identical between the two png_resize runs")

## API time

From the log of each run. `api_time_s` is the time between sending the request and receiving the response headers. A run is a single pass over 50 images, so the two identical `png_resize` runs show how much the time varies between runs, and any difference smaller than that is not significant.

In [ ]:
def read_log(name):
    with (EXPERIMENTS_DIR / name / "log.csv").open() as f:
        return list(csv.DictReader(f))


print(f"{'run':<18}{'retries':>8}{'median':>8}{'mean':>8}{'p90':>8}{'response KB':>13}")
for name in RUNS:
    rows = read_log(name)
    times = sorted(float(row["api_time_s"]) for row in rows)
    response_kb = statistics.mean(int(row["response_bytes"]) for row in rows) / 1000
    print(
        f"{name:<18}{sum(int(row['retries']) for row in rows):>8}"
        f"{statistics.median(times):>8.2f}{statistics.mean(times):>8.2f}"
        f"{times[int(0.9 * len(times))]:>8.2f}{response_kb:>13.1f}"
    )

## Conclusions

On 2026-10-02, with the executions of this analysis on the 50 images:

* **The model is deterministic.** Two runs with the same options gave identical masks, and the unresized run of this execution gave exactly the masks of the run made the day before, so differences between runs come from the changes made to the input, not from noise or drift of the endpoint.
* **Resizing costs a little accuracy.** With a lossless PNG, the clothes mIoU over the dataset went from 0.8865 to 0.8848 (-0.0017), and the dataset mIoU of all classes from 0.6519 to 0.6495 (-0.0024). 0.47% of the pixels changed class, and the clothes mIoU is lower on 40 of the 50 images. The model resizes the image itself, so the script's resampling and the upsampling of the mask back to the original size probably explain it, but this was not isolated.
* **JPEG costs more.** The clothes mIoU went down to 0.8804 (-0.0061), the mIoU of all classes to 0.6451 (-0.0068), about two thirds of it from the compression itself. Small classes like the legs and hair lost the most, the clothing classes less.
* **The payload gets much smaller.** The JPEG payload is about an eighth of the original, the PNG resize about two thirds.
* **The API time improves a little.** Over two executions of the script, the median API time was 0.91 to 0.97 s unresized, 0.74 to 0.89 s for the PNG resize and 0.72 to 0.74 s with JPEG. The two identical PNG resize runs differ by 0.04 to 0.05 s, which is the noise level: the gain of the resize (about 0.12 s in both executions) is above it, but JPEG against PNG is not distinguishable. Single passes over 50 images only give an indication.

The options are therefore disabled by default. `--resize` is the cheaper compromise when bandwidth matters, `--jpeg` only when it matters a lot. If a new execution gives different figures, the printed results above take precedence over this text.